# AKILAN PDF Artifact Workbench

This notebook discovers complex PDFs placed under `data/`, builds deterministic AKILAN artifacts, and produces a compact validation summary. PDF binaries remain local and are not committed by default.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from akilan import ExtractionConfig, PDFArtifactBuilder


In [ ]:
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / 'data'
ARTIFACT_DIR = REPO_ROOT / 'artifacts' / 'notebook'
PDFS = sorted(DATA_DIR.rglob('*.pdf')) if DATA_DIR.exists() else []
print(f'Discovered {len(PDFS)} PDF(s) under {DATA_DIR}')
for pdf in PDFS:
    print('-', pdf.relative_to(REPO_ROOT))


## Build artifacts

Page rendering is disabled by default to keep the workbench efficient. Enable it when visual comparison is required.

In [ ]:
config = ExtractionConfig(
    render_pages=False,
    include_characters=False,
    overwrite=True,
)
builder = PDFArtifactBuilder(config)
results = []
for pdf in PDFS:
    output_dir = ARTIFACT_DIR / pdf.stem
    try:
        artifact = builder.build(pdf, output_dir)
        results.append({
            'pdf': str(pdf.relative_to(REPO_ROOT)),
            'status': 'passed',
            'output': str(output_dir.relative_to(REPO_ROOT)),
            'statistics': artifact.statistics,
            'error': None,
        })
    except Exception as exc:
        results.append({
            'pdf': str(pdf.relative_to(REPO_ROOT)),
            'status': 'failed',
            'output': str(output_dir.relative_to(REPO_ROOT)),
            'statistics': {},
            'error': f'{type(exc).__name__}: {exc}',
        })
results


## Persist validation report

The report is suitable for attaching to a GitHub issue or converting into a focused `user_feedback.md` entry.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
report_path = ARTIFACT_DIR / 'validation_report.json'
report_path.write_text(json.dumps(results, indent=2, default=str) + '\n', encoding='utf-8')
print(f'Wrote {report_path}')
print(f'Passed: {sum(row["status"] == "passed" for row in results)}')
print(f'Failed: {sum(row["status"] == "failed" for row in results)}')
